https://chatgpt.com/g/g-p-68d161e5e43c81918b404ba2411ad90e-slk-correlaid/c/68fe80cf-0180-832a-9a22-e1ba939d400a

In [1]:
# %pip install requests tqdm
# %pip install hda -U

In [2]:
import os
import sys
from pathlib import Path
from typing import Dict, List, Tuple

from hda import Client, Configuration
from tqdm import tqdm


In [3]:
# Kosovo bbox in EPSG:4326 [minLon, minLat, maxLon, maxLat]
BBOX = [20.0, 41.8, 21.8, 43.3]

# Output root
# OUT_DIR = Path("/Users/vlad/Library/CloudStorage/GoogleDrive-vladimir.smirnov@rhodaris.com/.shortcut-targets-by-id/1p4PDClIbV5Br-EkGzuvhH5WVU1BudYkH/SLK CorrelAid 2025/Copernicus/batchimport")
OUT_DIR = Path("/Users/vlad/Downloads/hda_test_hrl_tc")

# Datasets and their temporal parameters
# For CHANGE products, HRL uses periods like "2012-2015", "2015-2018", "2018-2023".
DATASETS: Dict[str, Dict] = {
    # Tree Cover Density (numeric % per pixel)
    "EO:EEA:DAT:HRL:TCF": {
        "label": "TCD",
        "years": ["2012", "2015", "2018", "2019", "2020","2021","2022", "2023"],
        "extra": {"resolution": "10m", "productType": "Tree Cover Density", "itemsPerPage": 200, "startIndex": 0}
    },
    # Confidence layer for TCD
    "EO:EEA:DAT:HRL:TCF": {
        "label": "TCD_CONF",
        "years": ["2012", "2015", "2018", "2019", "2020","2021","2022", "2023"],
        "extra": {"resolution": "10m", "productType": "Tree Cover Density Confidence Layer"}
    },
    # Tree Cover Presence Change
    "EO:EEA:DAT:HRL:TCF": {
        "label": "TCD_CHANGE",
        "periods": ["2012-2015", "2015-2018", "2018-2021"],
        "extra": {"resolution": "20m", "productType": "Tree Cover Presence Change"}
    },
    # Confidence layer for Change
    "EO:EEA:DAT:HRL:TCF": {
        "label": "TCD_CHANGE_CONF",
        "periods": ["2012-2015", "2015-2018", "2018-2021"],
        "extra": {"resolution": "20m", "productType": "Tree Cover Presence Change Confidence Layer"}
    },
}

# HDA pagination
ITEMS_PER_PAGE = 200
START_INDEX = 0

# Toggle automatic merge with GDAL after download (per group)
RUN_GDAL_MERGE = False  # set to True if you want automatic mosaics
GDAL_OUTPUT_CRS = "EPSG:3857"  # used only in the example warp step below

In [4]:
def ensure_outdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

In [5]:
def hda_client_or_exit() -> Client:
    hdarc = Path(Path.home()/'.hdarc')
    if not hdarc.is_file():
        import getpass
        USERNAME = input('Enter your username: ')
        PASSWORD = getpass.getpass('Enter your password: ')

        with open(Path.home()/'.hdarc', 'w') as f:
            f.write(f'user: {USERNAME}\n')
            f.write(f'password:{PASSWORD}\n')
    else:
        print('Configuration file already exists.')

    try:
        client = Client()
        client.metadata(dataset_id="EO:EEA:DAT:HRL:TCF")  # Test connection
        print(f"Logged in as {client.config.user}")
        return client
    except Exception as e:
        print("Failed to initialize HDA Client. Check your .hdarc file.")
        raise

In [89]:
def page_search(client: Client, query: dict) -> list:
    print(f"Searching {query['dataset_id']} with filter {query}")
    try:
        results = client.search(
            dataset_id=query["dataset_id"],            
            productType=query.get("productType"),
            resolution=query.get("resolution"),
            year=query.get("year"),
            bbox=query["bbox"],
            itemsPerPage=query.get("itemsPerPage"),
            startIndex=query.get("startIndex")
        )
        print(f"  → {len(results)} products found")
        return results
    except Exception as e:
        print(f"Error during search: {e}")
        return []

In [9]:
def download_matches(matches: list, out_dir: Path):
    ensure_outdir(out_dir)
    for prd in tqdm(matches, desc=f"Downloading to {out_dir}", unit="tile"):
        try:
            prd.download(str(out_dir))
        except Exception as e:
            print(f"Download failed for {getattr(prd, 'id', 'unknown')}: {e}")

In [6]:
hda_client = Client()
help(hda_client.metadata)

print(f"Logged in as {hda_client.config.user}")

Help on method metadata in module hda.api:

metadata(dataset_id) method of hda.api.Client instance
    Returns the metadata object for the given dataset.
    
    :param dataset_id: The dataset ID
    :type dataset_id: str

Logged in as vladcorrelaid


In [7]:
hda_client.metadata(dataset_id="EO:EEA:DAT:HRL:TCF")  # Test connection
# hda_client.metadata(dataset_id="EO:EEA:DAT:CLMS_HRVPP_VPP")  # Test connection

{'type': 'object',
 'title': 'Queryable',
 'properties': {'dataset_id': {'title': 'Dataset_id',
   'type': 'string',
   'oneOf': [{'const': 'EO:EEA:DAT:HRL:TCF',
     'title': 'EO:EEA:DAT:HRL:TCF',
     'group': None}]},
  'bbox': {'title': 'bbox',
   'type': 'array',
   'minItems': 4,
   'maxItems': 4,
   'items': [{'type': 'number', 'maximum': 180, 'minimum': -180},
    {'type': 'number', 'maximum': 90, 'minimum': -90},
    {'type': 'number', 'maximum': 180, 'minimum': -180},
    {'type': 'number', 'maximum': 90, 'minimum': -90}]},
  'productType': {'title': 'Product Type',
   'type': 'string',
   'oneOf': [{'const': 'Broadleaved Cover Density',
     'title': 'Broadleaved Cover Density',
     'group': None},
    {'const': 'Coniferous Cover Density',
     'title': 'Coniferous Cover Density',
     'group': None},
    {'const': 'Dominant Leaf Type',
     'title': 'Dominant Leaf Type',
     'group': None},
    {'const': 'Dominant Leaf Type Change',
     'title': 'Dominant Leaf Type Chang

In [10]:
ensure_outdir(OUT_DIR)
client = hda_client_or_exit()

# Test with one dataset
dataset_id = "EO:EEA:DAT:HRL:TCF"  # Example: Tree Cover Density
cfg = DATASETS[dataset_id]
label = "TCD_CONF"
#periods 20m TCD_CHANGE - Tree Cover Presence Change
#periods 20m TCD_CHANGE_CONF - Tree Cover Presence Change Confidence Layer
years = [2012, 2013, 2015, 2018, 2019, 2020, 2021, 2022, 2023]
periods = ["2012-2015", "2015-2018", "2018-2021"]

# Iterate through all years in the dataset configuration
for year in years:
    # print(f"\n=== Testing {label} {year} ===")
    print(year)
    out = OUT_DIR / label / str(year)
    ensure_outdir(out)

    query = {
        "dataset_id": dataset_id,  # Required
        "productType": "Forest Type",  # Matches allowed values
        "resolution": "10m",  # Matches allowed values
        "year": str(year),  # Matches allowed values
        "bbox": [20.0, 41.8, 21.8, 43.3],  # Valid bbox
        "itemsPerPage": 200,  # Matches pattern
        "startIndex": 0  # Matches pattern
    }

    # Search and download products for the current year
    products = client.search(query)
    download_matches(products, out)

print("\n✅ Test complete. Files saved under:", OUT_DIR.resolve())

Configuration file already exists.
Logged in as vladcorrelaid
2012


2013


2015


2018


2019


2020


2021


2022


2023



✅ Test complete. Files saved under: /Users/vlad/Downloads/hda_test_hrl_tc
